<a href="https://colab.research.google.com/github/SujaAK/HindiASR/blob/main/5Ksize_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers -q
!pip install snac google-generativeai deepgram-sdk -q
!pip install datasets huggingface_hub soundfile jiwer -q

In [ ]:
!pip install torch==2.11.0+cu128 torchaudio==2.11.0+cu128 torchvision==0.26.0+cu128 --index-url https://download.pytorch.org/whl/cu128 -q

In [ ]:
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
from huggingface_hub import login
login()

import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
gemini_model = genai.GenerativeModel("gemini-2.5-flash")

# Quick test
response = gemini_model.generate_content("Say hello in one word.")
print(response.text)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from snac import SNAC

print("Loading Veena model...")
model = AutoModelForCausalLM.from_pretrained(
    "maya-research/veena-tts",
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
tokenizer = AutoTokenizer.from_pretrained("maya-research/veena-tts", trust_remote_code=True)

print("Loading SNAC decoder...")
snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").eval().cuda()

print("All models loaded successfully!")

In [ ]:
START_OF_SPEECH_TOKEN = 128257
END_OF_SPEECH_TOKEN = 128258
START_OF_HUMAN_TOKEN = 128259
END_OF_HUMAN_TOKEN = 128260
START_OF_AI_TOKEN = 128261
END_OF_AI_TOKEN = 128262
AUDIO_CODE_BASE_OFFSET = 128266

def generate_speech(text, speaker="kavya", temperature=0.4, top_p=0.9):
    prompt = f"<spk_{speaker}> {text}"
    prompt_tokens = tokenizer.encode(prompt, add_special_tokens=False)

    input_tokens = [
        START_OF_HUMAN_TOKEN, *prompt_tokens, END_OF_HUMAN_TOKEN,
        START_OF_AI_TOKEN, START_OF_SPEECH_TOKEN
    ]
    input_ids = torch.tensor([input_tokens], device=model.device)
    max_tokens = min(int(len(text) * 1.3) * 7 + 21, 700)

    with torch.no_grad():
        output = model.generate(
            input_ids, max_new_tokens=max_tokens, do_sample=True,
            temperature=temperature, top_p=top_p, repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=[END_OF_SPEECH_TOKEN, END_OF_AI_TOKEN]
        )

    generated_ids = output[0][len(input_tokens):].tolist()
    snac_tokens = [t for t in generated_ids if AUDIO_CODE_BASE_OFFSET <= t < (AUDIO_CODE_BASE_OFFSET + 7*4096)]
    if not snac_tokens:
        raise ValueError("No audio tokens generated")
    return decode_snac_tokens(snac_tokens, snac_model)

def decode_snac_tokens(snac_tokens, snac_model):
    if not snac_tokens or len(snac_tokens) % 7 != 0:
        return None
    snac_device = next(snac_model.parameters()).device
    codes_lvl = [[] for _ in range(3)]
    offsets = [AUDIO_CODE_BASE_OFFSET + i * 4096 for i in range(7)]

    for i in range(0, len(snac_tokens), 7):
        codes_lvl[0].append(snac_tokens[i] - offsets[0])
        codes_lvl[1].append(snac_tokens[i+1] - offsets[1])
        codes_lvl[1].append(snac_tokens[i+4] - offsets[4])
        codes_lvl[2].append(snac_tokens[i+2] - offsets[2])
        codes_lvl[2].append(snac_tokens[i+3] - offsets[3])
        codes_lvl[2].append(snac_tokens[i+5] - offsets[5])
        codes_lvl[2].append(snac_tokens[i+6] - offsets[6])

    hierarchical_codes = []
    for lvl in codes_lvl:
        t = torch.tensor(lvl, dtype=torch.int32, device=snac_device).unsqueeze(0)
        hierarchical_codes.append(t)

    with torch.no_grad():
        audio_hat = snac_model.decode(hierarchical_codes)
    return audio_hat.squeeze().clamp(-1, 1).cpu().numpy()

print("Helper functions ready.")

In [ ]:
import soundfile as sf
import IPython.display as ipd

text_hindi = "नमस्ते, यह एक परीक्षण वाक्य है"
audio = generate_speech(text_hindi, speaker="kavya")
sf.write("test_veena.wav", audio, 24000)
display(ipd.Audio("test_veena.wav"))

In [ ]:
import re
import time
import json
import os

DOMAIN_PROMPTS = {
    "dairy_order": "एक ग्राहक डेयरी की दुकान को फोन पर दूध, दही, पनीर या घी का ऑर्डर दे रहा है।",
    "payment": "एक ग्राहक या दुकानदार पेमेंट, बिल, चालान या क्रेडिट नोट के बारे में बात कर रहा है।",
    "delivery": "डिलीवरी का समय, लोकेशन या टाइमिंग बदलने को लेकर बातचीत हो रही है।",
    "order": "नया ऑर्डर देना, ऑर्डर कैंसिल करना या रिपीट ऑर्डर को लेकर बातचीत हो रही है।",
    "general": "स्टॉक, रेट, क्वालिटी कंप्लेंट या दुकान बंद होने को लेकर सामान्य बातचीत हो रही है।",
}

SYSTEM_INSTRUCTION = (
    "आप एक भारतीय छोटे व्यापारी (दुकानदार) और उसके ग्राहक के बीच होने वाली स्वाभाविक, "
    "बोलचाल की फोन कॉल बातचीत के वाक्य लिखते हैं। हमेशा देवनागरी लिपि में लिखें, "
    "अंग्रेज़ी के शब्द भी देवनागरी में लिखें (जैसे 'पेमेंट', 'डिलीवरी'), कभी भी लैटिन/रोमन लिपि "
    "का उपयोग न करें। संख्याएं हमेशा शब्दों में लिखें, अंकों में नहीं (जैसे 'दस' न कि '10')। "
    "हर वाक्य अलग, प्राकृतिक और विविध होना चाहिए — अलग-अलग नाम, मात्रा, समय और स्थिति का उपयोग करें, "
    "दोहराव से पूरी तरह बचें।"
)

hinglish_markers = ["पेमेंट", "डिलीवरी", "कन्फर्म", "स्टॉक", "क्वालिटी", "कंप्लेंट",
                     "चालान", "क्रेडिट", "कैंसिल", "रिपीट", "अवेलेबल"]

CHECKPOINT_FILE = "gemini_dataset_checkpoint.json"


def is_devanagari_only(text):
    return not re.search(r'[a-zA-Z]', text)


def tag_language_mix(sentence):
    return "hinglish" if any(m in sentence for m in hinglish_markers) else "hindi"


def load_checkpoint():
    """Load existing progress if a checkpoint file exists."""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
            records = json.load(f)
        print(f"Resuming from checkpoint: {len(records)} sentences already generated.")
        return records
    return []


def save_checkpoint(records):
    """Save current progress to disk."""
    with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)


def generate_batch_in_one_call(domain, count=50, max_retries=4):
    prompt = (
        f"{SYSTEM_INSTRUCTION}\n\n"
        f"विषय: {DOMAIN_PROMPTS[domain]}\n\n"
        f"{count} अलग-अलग, स्वाभाविक वाक्य लिखें। हर वाक्य एक नई लाइन पर लिखें। "
        f"कोई नंबरिंग, बुलेट पॉइंट, या अतिरिक्त टेक्स्ट न जोड़ें — सिर्फ वाक्य।"
    )
    for attempt in range(max_retries):
        try:
            response = gemini_model.generate_content(prompt)
            raw_lines = response.text.strip().split("\n")
            cleaned = []
            for line in raw_lines:
                line = line.strip()
                line = re.sub(r'^\d+[\.\)]\s*', '', line)
                line = line.strip('-* ').strip('"\'""\'\'')
                line = re.sub(r'\d+', '', line).strip()
                if line and len(line.split()) >= 3:
                    cleaned.append(line)
            return cleaned
        except Exception as e:
            wait = 30 * (attempt + 1)
            print(f"  Retry {attempt+1}/{max_retries} for '{domain}': {e}")
            print(f"  Waiting {wait}s...")
            time.sleep(wait)
    return []


def generate_gemini_dataset_at_scale(target_total=5000, per_call_count=50,
                                       delay_between_calls=15, checkpoint_every=5):
    """
    Generates sentences in batches, saving progress to disk every
    `checkpoint_every` API calls. If interrupted (rate limit, crash,
    disconnect), re-running this function picks up where it left off.
    """
    all_records = load_checkpoint()
    seen = set(r["transcript"] for r in all_records)
    domains = list(DOMAIN_PROMPTS.keys())
    call_num = 0

    while len(all_records) < target_total:
        domain = domains[call_num % len(domains)]
        call_num += 1
        print(f"[Call {call_num}] '{domain}' | Total so far: {len(all_records)}/{target_total}")

        sentences = generate_batch_in_one_call(domain, count=per_call_count)
        added = 0
        for sentence in sentences:
            if sentence in seen or not is_devanagari_only(sentence):
                continue
            seen.add(sentence)
            all_records.append({
                "transcript": sentence,
                "domain_tag": domain,
                "language_mix": tag_language_mix(sentence),
                "sentence_length": len(sentence.split()),
                "source": "gemini"
            })
            added += 1
        print(f"  -> Added {added} unique sentences.")

        if call_num % checkpoint_every == 0:
            save_checkpoint(all_records)
            print(f"  [Checkpoint saved: {len(all_records)} sentences]")

        time.sleep(delay_between_calls)

    save_checkpoint(all_records)
    print(f"\nDone. Final checkpoint saved: {len(all_records)} sentences.")
    return all_records[:target_total]


print("Function ready. Checkpoint file:", CHECKPOINT_FILE)
print("If this run gets interrupted (rate limit etc), just re-run this generation")
print("call again later - it will resume from the last saved checkpoint automatically.")

In [ ]:
gemini_dataset = generate_gemini_dataset_at_scale(
    target_total=5000,
    per_call_count=50,
    delay_between_calls=35,
    checkpoint_every=5
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = "/content/drive/MyDrive/HindiASR_Dataset"
print("Drive mounted. Checking saved files:")
print(os.listdir(SAVE_DIR))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = "/content/drive/MyDrive/HindiASR_Dataset"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Saving to: {SAVE_DIR}")

In [ ]:
CHECKPOINT_FILE = f"{SAVE_DIR}/gemini_dataset_checkpoint.json"
AUDIO_CHECKPOINT_FILE = f"{SAVE_DIR}/audio_generation_checkpoint.json"
os.makedirs(f"{SAVE_DIR}/generated_audio", exist_ok=True)

In [ ]:
import shutil

# Move your existing Gemini checkpoint into the persistent Drive folder
if os.path.exists("gemini_dataset_checkpoint.json"):
    shutil.copy("gemini_dataset_checkpoint.json", f"{SAVE_DIR}/gemini_dataset_checkpoint.json")
    print("Existing 531-sentence checkpoint copied to Drive.")

# Update paths going forward to use Drive
CHECKPOINT_FILE = f"{SAVE_DIR}/gemini_dataset_checkpoint.json"
AUDIO_CHECKPOINT_FILE = f"{SAVE_DIR}/audio_generation_checkpoint.json"
AUDIO_DIR = f"{SAVE_DIR}/generated_audio"
os.makedirs(AUDIO_DIR, exist_ok=True)

print(f"Checkpoint file: {CHECKPOINT_FILE}")
print(f"Audio checkpoint: {AUDIO_CHECKPOINT_FILE}")
print(f"Audio directory: {AUDIO_DIR}")

In [ ]:
import os

# Verify the checkpoint file actually exists and is readable from Drive
print("File exists in Drive:", os.path.exists(CHECKPOINT_FILE))
print("File size:", os.path.getsize(CHECKPOINT_FILE), "bytes")

# Actually load it back and confirm sentence count
import json
with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
    verify_data = json.load(f)
print("Sentences confirmed in Drive checkpoint:", len(verify_data))
print("Sample entry:", verify_data[0])

In [ ]:
import re
import time
import json
import os

DOMAIN_PROMPTS = {
    "dairy_order": "एक ग्राहक डेयरी की दुकान को फोन पर दूध, दही, पनीर या घी का ऑर्डर दे रहा है।",
    "payment": "एक ग्राहक या दुकानदार पेमेंट, बिल, चालान या क्रेडिट नोट के बारे में बात कर रहा है।",
    "delivery": "डिलीवरी का समय, लोकेशन या टाइमिंग बदलने को लेकर बातचीत हो रही है।",
    "order": "नया ऑर्डर देना, ऑर्डर कैंसिल करना या रिपीट ऑर्डर को लेकर बातचीत हो रही है।",
    "general": "स्टॉक, रेट, क्वालिटी कंप्लेंट या दुकान बंद होने को लेकर सामान्य बातचीत हो रही है।",
}

SYSTEM_INSTRUCTION = (
    "आप एक भारतीय छोटे व्यापारी (दुकानदार) और उसके ग्राहक के बीच होने वाली स्वाभाविक, "
    "बोलचाल की फोन कॉल बातचीत के वाक्य लिखते हैं। हमेशा देवनागरी लिपि में लिखें, "
    "अंग्रेज़ी के शब्द भी देवनागरी में लिखें (जैसे 'पेमेंट', 'डिलीवरी'), कभी भी लैटिन/रोमन लिपि "
    "का उपयोग न करें। संख्याएं हमेशा शब्दों में लिखें, अंकों में नहीं (जैसे 'दस' न कि '10')। "
    "हर वाक्य अलग, प्राकृतिक और विविध होना चाहिए — अलग-अलग नाम, मात्रा, समय और स्थिति का उपयोग करें, "
    "दोहराव से पूरी तरह बचें।"
)

hinglish_markers = ["पेमेंट", "डिलीवरी", "कन्फर्म", "स्टॉक", "क्वालिटी", "कंप्लेंट",
                     "चालान", "क्रेडिट", "कैंसिल", "रिपीट", "अवेलेबल"]

CHECKPOINT_FILE = f"{SAVE_DIR}/gemini_dataset_checkpoint.json"

def is_devanagari_only(text):
    return not re.search(r'[a-zA-Z]', text)

def tag_language_mix(sentence):
    return "hinglish" if any(m in sentence for m in hinglish_markers) else "hindi"

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
            records = json.load(f)
        print(f"Resuming from checkpoint: {len(records)} sentences already generated.")
        return records
    return []

def save_checkpoint(records):
    with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

def generate_batch_in_one_call(domain, count=50, max_retries=4):
    prompt = (
        f"{SYSTEM_INSTRUCTION}\n\n"
        f"विषय: {DOMAIN_PROMPTS[domain]}\n\n"
        f"{count} अलग-अलग, स्वाभाविक वाक्य लिखें। हर वाक्य एक नई लाइन पर लिखें। "
        f"कोई नंबरिंग, बुलेट पॉइंट, या अतिरिक्त टेक्स्ट न जोड़ें — सिर्फ वाक्य।"
    )
    for attempt in range(max_retries):
        try:
            response = gemini_model.generate_content(prompt)
            raw_lines = response.text.strip().split("\n")
            cleaned = []
            for line in raw_lines:
                line = line.strip()
                line = re.sub(r'^\d+[\.\)]\s*', '', line)
                line = line.strip('-* ').strip('"\'""\'\'')
                line = re.sub(r'\d+', '', line).strip()
                if line and len(line.split()) >= 3:
                    cleaned.append(line)
            return cleaned
        except Exception as e:
            wait = 40 * (attempt + 1)
            print(f"  Retry {attempt+1}/{max_retries} for '{domain}': {e}")
            print(f"  Waiting {wait}s...")
            time.sleep(wait)
    return []

def generate_gemini_dataset_at_scale(target_total=5000, per_call_count=50,
                                       delay_between_calls=35, checkpoint_every=5):
    all_records = load_checkpoint()
    seen = set(r["transcript"] for r in all_records)
    domains = list(DOMAIN_PROMPTS.keys())
    call_num = 0

    while len(all_records) < target_total:
        domain = domains[call_num % len(domains)]
        call_num += 1
        print(f"[Call {call_num}] '{domain}' | Total so far: {len(all_records)}/{target_total}")

        sentences = generate_batch_in_one_call(domain, count=per_call_count)
        added = 0
        for sentence in sentences:
            if sentence in seen or not is_devanagari_only(sentence):
                continue
            seen.add(sentence)
            all_records.append({
                "transcript": sentence,
                "domain_tag": domain,
                "language_mix": tag_language_mix(sentence),
                "sentence_length": len(sentence.split()),
                "source": "gemini"
            })
            added += 1
        print(f"  -> Added {added} unique sentences.")

        if call_num % checkpoint_every == 0:
            save_checkpoint(all_records)
            print(f"  [Checkpoint saved to Drive: {len(all_records)} sentences]")

        time.sleep(delay_between_calls)

    save_checkpoint(all_records)
    print(f"\nDone. Final checkpoint saved: {len(all_records)} sentences.")
    return all_records[:target_total]

print("Function ready, pointed at Drive checkpoint.")

In [ ]:
gemini_dataset = generate_gemini_dataset_at_scale(
    target_total=5000,
    per_call_count=50,
    delay_between_calls=35,
    checkpoint_every=5
)

In [ ]:
import json

CHECKPOINT_FILE = f"{SAVE_DIR}/gemini_dataset_checkpoint.json"

with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
    saved_data = json.load(f)

print(f"Confirmed: {len(saved_data)} sentences currently saved in Drive checkpoint.")

In [ ]:
import os
import gc
import time
import soundfile as sf
import torch

AUDIO_DIR = f"{SAVE_DIR}/generated_audio"
AUDIO_CHECKPOINT_FILE = f"{SAVE_DIR}/audio_generation_checkpoint.json"
os.makedirs(AUDIO_DIR, exist_ok=True)

speakers = ["kavya", "agastya", "maitri", "vinaya"]


def load_audio_checkpoint():
    if os.path.exists(AUDIO_CHECKPOINT_FILE):
        with open(AUDIO_CHECKPOINT_FILE, "r", encoding="utf-8") as f:
            records = json.load(f)
        print(f"Resuming audio generation: {len(records)} samples already done.")
        return records
    return []


def save_audio_checkpoint(records):
    with open(AUDIO_CHECKPOINT_FILE, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)


def generate_audio_batch(sentences, clear_every=50):
    audio_records = load_audio_checkpoint()
    done_texts = set(r["transcript"] for r in audio_records)
    failed = []

    for i, record in enumerate(sentences):
        if record["transcript"] in done_texts:
            continue

        text = record["transcript"]
        speaker = speakers[i % len(speakers)]

        try:
            audio = generate_speech(text, speaker=speaker)
            filename = f"{AUDIO_DIR}/sample_{i:05d}.wav"
            sf.write(filename, audio, 24000)

            audio_records.append({
                "audio": filename,
                "transcript": text,
                "domain_tag": record["domain_tag"],
                "language_mix": record["language_mix"],
                "sentence_length": record["sentence_length"],
                "speaker": speaker,
            })

        except Exception as e:
            print(f"  FAILED sample {i} ('{text[:30]}...'): {e}")
            failed.append({"index": i, "text": text, "error": str(e)})
            continue

        if (i + 1) % 10 == 0:
            print(f"  Progress: {len(audio_records)} audio samples generated...")

        if (i + 1) % clear_every == 0:
            torch.cuda.empty_cache()
            gc.collect()
            save_audio_checkpoint(audio_records)
            print(f"  [GPU cleared + checkpoint saved to Drive at {len(audio_records)} samples]")

    save_audio_checkpoint(audio_records)
    print(f"\nDone! {len(audio_records)} succeeded, {len(failed)} failed.")
    return audio_records, failed


# Run on your 1,545 sentences
audio_records, failed_samples = generate_audio_batch(gemini_dataset, clear_every=50)

In [ ]:
import json

CHECKPOINT_FILE = f"{SAVE_DIR}/gemini_dataset_checkpoint.json"

with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
    gemini_dataset = json.load(f)

print(f"Loaded {len(gemini_dataset)} sentences for audio generation")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
SAVE_DIR = "/content/drive/MyDrive/HindiASR_Dataset"
AUDIO_CHECKPOINT_FILE = f"{SAVE_DIR}/audio_generation_checkpoint.json"

with open(AUDIO_CHECKPOINT_FILE, "r", encoding="utf-8") as f:
    audio_records = json.load(f)

print(f"Confirmed: {len(audio_records)} audio samples safely saved in Drive.")

In [ ]:
import json
import os
import gc
import soundfile as sf

with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
    gemini_dataset = json.load(f)
print(f"Loaded {len(gemini_dataset)} sentences")

speakers = ["kavya", "agastya", "maitri", "vinaya"]

def load_audio_checkpoint():
    if os.path.exists(AUDIO_CHECKPOINT_FILE):
        with open(AUDIO_CHECKPOINT_FILE, "r", encoding="utf-8") as f:
            records = json.load(f)
        print(f"Resuming audio generation: {len(records)} samples already done.")
        return records
    return []

def save_audio_checkpoint(records):
    with open(AUDIO_CHECKPOINT_FILE, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

def generate_audio_batch(sentences, clear_every=50):
    audio_records = load_audio_checkpoint()
    done_texts = set(r["transcript"] for r in audio_records)
    failed = []

    for i, record in enumerate(sentences):
        if record["transcript"] in done_texts:
            continue

        text = record["transcript"]
        speaker = speakers[i % len(speakers)]

        try:
            audio = generate_speech(text, speaker=speaker)
            filename = f"{AUDIO_DIR}/sample_{i:05d}.wav"
            sf.write(filename, audio, 24000)

            audio_records.append({
                "audio": filename, "transcript": text,
                "domain_tag": record["domain_tag"], "language_mix": record["language_mix"],
                "sentence_length": record["sentence_length"], "speaker": speaker,
            })
        except Exception as e:
            print(f"  FAILED sample {i}: {e}")
            failed.append({"index": i, "text": text, "error": str(e)})
            continue

        if (i + 1) % 10 == 0:
            print(f"  Progress: {len(audio_records)} audio samples generated...")
        if (i + 1) % clear_every == 0:
            torch.cuda.empty_cache()
            gc.collect()
            save_audio_checkpoint(audio_records)
            print(f"  [Checkpoint saved at {len(audio_records)} samples]")

    save_audio_checkpoint(audio_records)
    print(f"\nDone! {len(audio_records)} succeeded, {len(failed)} failed.")
    return audio_records, failed

audio_records, failed_samples = generate_audio_batch(gemini_dataset, clear_every=50)

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/HindiASR_Dataset"
AUDIO_DIR = f"{SAVE_DIR}/generated_audio"
AUDIO_CHECKPOINT_FILE = f"{SAVE_DIR}/audio_generation_checkpoint.json"
CHECKPOINT_FILE = f"{SAVE_DIR}/gemini_dataset_checkpoint.json"

import json
import os
import gc
import soundfile as sf

with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
    gemini_dataset = json.load(f)
print(f"Loaded {len(gemini_dataset)} sentences")

In [ ]:
!pip install -U datasets huggingface_hub soundfile -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/HindiASR_Dataset"
AUDIO_DIR = f"{SAVE_DIR}/generated_audio"
AUDIO_CHECKPOINT_FILE = f"{SAVE_DIR}/audio_generation_checkpoint.json"

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import json
from datasets import Dataset, Audio

with open(AUDIO_CHECKPOINT_FILE, "r", encoding="utf-8") as f:
    audio_records = json.load(f)

print(f"Loaded {len(audio_records)} samples")
print("Sample entry:", audio_records[0])

hf_dataset = Dataset.from_list(audio_records)
hf_dataset = hf_dataset.cast_column("audio", Audio(sampling_rate=24000))

print(hf_dataset)

In [ ]:
hf_dataset.push_to_hub("shujaAK/hindi-dairy-dataset-veena")
print("Pushed to HuggingFace successfully!")

In [ ]:
import json
from datasets import Dataset, Audio

AUDIO_CHECKPOINT_FILE = f"{SAVE_DIR}/audio_generation_checkpoint.json"

with open(AUDIO_CHECKPOINT_FILE, "r", encoding="utf-8") as f:
    audio_records = json.load(f)

print(f"Total samples in checkpoint right now: {len(audio_records)}")

hf_dataset = Dataset.from_list(audio_records)
hf_dataset = hf_dataset.cast_column("audio", Audio(sampling_rate=24000))

In [ ]:
!pip install deepgram-sdk -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/HindiASR_Dataset"
AUDIO_CHECKPOINT_FILE = f"{SAVE_DIR}/audio_generation_checkpoint.json"
DEEPGRAM_CHECKPOINT_FILE = f"{SAVE_DIR}/deepgram_validation_checkpoint.json"

In [ ]:
import deepgram
print(deepgram.__version__)
print([x for x in dir(deepgram) if not x.startswith('_')])

In [ ]:
from google.colab import userdata
from deepgram import DeepgramClient

DEEPGRAM_API_KEY = userdata.get('DEEPGRAM_API_KEY')
client = DeepgramClient(api_key=DEEPGRAM_API_KEY)

def transcribe_with_deepgram(audio_path):
    with open(audio_path, "rb") as f:
        audio_bytes = f.read()
    response = client.listen.v1.media.transcribe_file(
        request=audio_bytes,
        model="nova-3",
        language="hi",
        smart_format=True,
    )
    return response.results.channels[0].alternatives[0].transcript

# Quick test on one file
import json
with open(AUDIO_CHECKPOINT_FILE, "r", encoding="utf-8") as f:
    audio_records = json.load(f)

test_result = transcribe_with_deepgram(audio_records[0]["audio"])
print("Original text:", audio_records[0]["transcript"])
print("Deepgram heard:", test_result)

In [ ]:
import os
import json
import time

DEEPGRAM_CHECKPOINT_FILE = f"{SAVE_DIR}/deepgram_validation_checkpoint.json"

def load_deepgram_checkpoint():
    if os.path.exists(DEEPGRAM_CHECKPOINT_FILE):
        with open(DEEPGRAM_CHECKPOINT_FILE, "r", encoding="utf-8") as f:
            records = json.load(f)
        print(f"Resuming Deepgram validation: {len(records)} samples already done.")
        return records
    return []

def save_deepgram_checkpoint(records):
    with open(DEEPGRAM_CHECKPOINT_FILE, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

def run_deepgram_validation(records, checkpoint_every=25):
    results = load_deepgram_checkpoint()
    done_audio = set(r["audio"] for r in results)
    failed = []

    for i, record in enumerate(records):
        if record["audio"] in done_audio:
            continue

        try:
            deepgram_text = transcribe_with_deepgram(record["audio"])
            results.append({
                **record,
                "deepgram_transcript": deepgram_text,
            })
        except Exception as e:
            print(f"  FAILED on sample {i} ('{record['audio']}'): {e}")
            failed.append({"index": i, "audio": record["audio"], "error": str(e)})
            time.sleep(2)
            continue

        if (i + 1) % 10 == 0:
            print(f"  Progress: {len(results)} validated...")

        if (i + 1) % checkpoint_every == 0:
            save_deepgram_checkpoint(results)
            print(f"  [Checkpoint saved at {len(results)} samples]")

    save_deepgram_checkpoint(results)
    print(f"\nDone! {len(results)} validated, {len(failed)} failed.")
    return results, failed


# Load your 850 audio samples and run validation
with open(AUDIO_CHECKPOINT_FILE, "r", encoding="utf-8") as f:
    audio_records = json.load(f)

print(f"Starting Deepgram validation on {len(audio_records)} samples...")
validated_records, failed_validations = run_deepgram_validation(audio_records, checkpoint_every=25)

In [ ]:
!pip install jiwer -q

In [ ]:
import jiwer

with open(DEEPGRAM_CHECKPOINT_FILE, "r", encoding="utf-8") as f:
    validated_records = json.load(f)

references = [r["transcript"] for r in validated_records]
predictions = [r["deepgram_transcript"] for r in validated_records]

# Overall aggregate WER/CER across all 850
overall_wer = jiwer.wer(references, predictions)
overall_cer = jiwer.cer(references, predictions)

print(f"Overall WER (850 samples): {overall_wer:.3f}")
print(f"Overall CER (850 samples): {overall_cer:.3f}")
print()

# Per-sample WER/CER, attached to each record
for r in validated_records:
    sample_wer = jiwer.wer(r["transcript"], r["deepgram_transcript"])
    sample_cer = jiwer.cer(r["transcript"], r["deepgram_transcript"])
    r["deepgram_wer"] = sample_wer
    r["deepgram_cer"] = sample_cer

# Save this enriched version back to the checkpoint (now includes per-sample WER/CER too)
save_deepgram_checkpoint(validated_records)

# Show some actual examples - best and worst matches
sorted_by_wer = sorted(validated_records, key=lambda x: x["deepgram_wer"])

print("=== 5 BEST MATCHES (lowest WER) ===")
for r in sorted_by_wer[:5]:
    print(f"WER: {r['deepgram_wer']:.3f}")
    print(f"  Gemini:   {r['transcript']}")
    print(f"  Deepgram: {r['deepgram_transcript']}")
    print()

print("=== 5 WORST MATCHES (highest WER) ===")
for r in sorted_by_wer[-5:]:
    print(f"WER: {r['deepgram_wer']:.3f}")
    print(f"  Gemini:   {r['transcript']}")
    print(f"  Deepgram: {r['deepgram_transcript']}")
    print()